# 03a — Balancing-authority territories (snapshot pinned)

Reinstated from the last committed pre-pivot notebook for the provenance repair. This version uses the fixed **HIFLD Control Areas Shapefile item modified 2021-12-08** (ArcGIS item `17499e6de9104f7288ce2ccc9239bc98`), not a mutable FeatureServer query. The raw ZIP is retained in staging and its SHA-256 is recorded.


In [1]:
from pathlib import Path
import hashlib
import json
import zipfile
from datetime import datetime, timezone

import geopandas as gpd
import pandas as pd
import requests

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
STAGING = PROJECT_ROOT / "data" / "staging"
STAGING.mkdir(parents=True, exist_ok=True)

HIFLD_ITEM_ID = "17499e6de9104f7288ce2ccc9239bc98"
HIFLD_SNAPSHOT_URL = f"https://www.arcgis.com/sharing/rest/content/items/{HIFLD_ITEM_ID}/data"
HIFLD_ITEM_MODIFIED_UTC = "2021-12-08T21:51:56Z"
zip_path = STAGING / "hifld_control_areas_2021-12-08_raw.zip"

response = requests.get(HIFLD_SNAPSHOT_URL, timeout=180)
response.raise_for_status()
zip_path.write_bytes(response.content)
raw_sha256 = hashlib.sha256(response.content).hexdigest()

extract_dir = STAGING / "hifld_control_areas_2021-12-08"
extract_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path) as archive:
    members = archive.namelist()
    archive.extractall(extract_dir)

shape_paths = sorted(extract_dir.rglob("*.shp"))
if len(shape_paths) != 1:
    raise RuntimeError(f"Expected one HIFLD shapefile, found {shape_paths}")
ba = gpd.read_file(shape_paths[0]).to_crs("EPSG:4326")
required_ba = {"ID", "NAME"}
missing_ba = required_ba.difference(ba.columns)
if missing_ba:
    raise KeyError(f"HIFLD snapshot missing required fields: {sorted(missing_ba)}")
ba.to_file(STAGING / "ba_territories.geojson", driver="GeoJSON")

plants = gpd.read_file(STAGING / "power_plants.geojson").to_crs("EPSG:4326")
plants["_source_row_id"] = range(len(plants))
plants["capacity_mw"] = pd.to_numeric(plants["nameplate-capacity-mw"], errors="coerce")
plants_ba = gpd.sjoin(
    plants,
    ba[["ID", "NAME", "geometry"]].rename(
        columns={"ID": "ba_boundary_id", "NAME": "ba_name"}
    ),
    how="left",
    predicate="within",
).drop(columns=["index_right"], errors="ignore")
overlap_match_count = int(plants_ba.duplicated("_source_row_id", keep=False).sum())
plants_ba = plants_ba.drop_duplicates("_source_row_id", keep="first")
plants_ba = plants_ba.drop(columns=["_source_row_id"])
plants_ba["ba_code"] = plants_ba["balancing_authority_code"]
plants_ba.to_file(STAGING / "power_plants_with_ba.geojson", driver="GeoJSON")

metadata = {
    "source": "HIFLD Control Areas, fixed ArcGIS Online Shapefile item",
    "arcgis_item_id": HIFLD_ITEM_ID,
    "arcgis_item_modified_utc": HIFLD_ITEM_MODIFIED_UTC,
    "snapshot_url": HIFLD_SNAPSHOT_URL,
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "raw_response_path": str(zip_path.relative_to(PROJECT_ROOT)),
    "raw_sha256": raw_sha256,
    "raw_bytes": len(response.content),
    "archive_members": members,
    "boundary_feature_count": len(ba),
    "plant_record_count": len(plants_ba),
    "boundary_matched_count": int(plants_ba["ba_boundary_id"].notna().sum()),
    "overlap_join_rows_before_deduplication": overlap_match_count,
    "eia_ba_code_present_count": int(plants_ba["ba_code"].notna().sum()),
    "derived_paths": [
        "data/staging/ba_territories.geojson",
        "data/staging/power_plants_with_ba.geojson",
    ],
    "superseded_live_service": (
        "https://services5.arcgis.com/bsqU0jSPAuI04L89/arcgis/rest/services/"
        "Balancing_Authorities/FeatureServer/0"
    ),
    "residual_unpinnable_risk": None,
}
(STAGING / "hifld_control_areas_2021-12-08_metadata.json").write_text(
    json.dumps(metadata, indent=2) + "\n"
)
print(json.dumps(metadata, indent=2))


{
  "source": "HIFLD Control Areas, fixed ArcGIS Online Shapefile item",
  "arcgis_item_id": "17499e6de9104f7288ce2ccc9239bc98",
  "arcgis_item_modified_utc": "2021-12-08T21:51:56Z",
  "snapshot_url": "https://www.arcgis.com/sharing/rest/content/items/17499e6de9104f7288ce2ccc9239bc98/data",
  "retrieved_at_utc": "2026-08-10T20:53:58.461030+00:00",
  "raw_response_path": "data/staging/hifld_control_areas_2021-12-08_raw.zip",
  "raw_sha256": "12296854e38d369ec1aef5eaf4f8eb5ca3bbfba1fa23029893ca1f4815a3c868",
  "raw_bytes": 23578889,
  "archive_members": [
    "Control_Areas.cpg",
    "Control_Areas.dbf",
    "Control_Areas.prj",
    "Control_Areas.shp",
    "Control_Areas.shp.xml",
    "Control_Areas.shx"
  ],
  "boundary_feature_count": 71,
  "plant_record_count": 25868,
  "boundary_matched_count": 25246,
  "overlap_join_rows_before_deduplication": 13661,
  "eia_ba_code_present_count": 25253,
  "derived_paths": [
    "data/staging/ba_territories.geojson",
    "data/staging/power_plants_